# Image Captioning Preset

Reusable Google Colab notebook template for image-captioning competitions.

This preset is modeled after a Thai image-captioning all-in-one workflow, but the paths, file names, model, prompt, metric, and training strategy are editable from the configuration cell.

Default flow:

1. Install libraries
2. Mount Drive or upload a zip
3. Extract and inspect files
4. Load annotations and sample submission
5. Run EDA
6. Build image-caption records
7. Create a fast pretrained-model baseline submission
8. Optionally fine-tune with LoRA/QLoRA
9. Tune decoding, evaluate, and generate submission
10. Optional improvement cells for stronger competition runs

In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================
# Colab-friendly installs. Re-run this cell after changing optional features.
import sys
import subprocess

INSTALL_LIBS = True

if INSTALL_LIBS:
    packages = [
        "transformers",
        "accelerate",
        "datasets",
        "peft",
        "bitsandbytes",
        "sentencepiece",
        "sacrebleu",
        "evaluate",
        "pandas",
        "pyarrow",
        "seaborn",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping installs.")

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
from pathlib import Path
import os
import re
import gc
import json
import math
import shutil
import random
import zipfile
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from transformers import PaliGemmaForConditionalGeneration
except Exception:
    PaliGemmaForConditionalGeneration = None

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
except Exception:
    LoraConfig = get_peft_model = prepare_model_for_kbit_training = PeftModel = None

import sacrebleu

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# ============================================================
# 3. Configuration placeholders
# ============================================================
# TODO: Edit this cell for each competition.

TASK_TYPE = "image_captioning"
RANDOM_STATE = 42
seed_everything(RANDOM_STATE)

# ----- Colab / data source -----
USE_GOOGLE_DRIVE = True
MOUNT_DRIVE = True
PROJECT_ROOT = Path("/content/drive/MyDrive/image_captioning_preset")

# If using a zip, set DATA_ZIP_PATH. If files are already extracted, set DATA_DIR directly.
DATA_ZIP_PATH = PROJECT_ROOT / "dataset.zip"   # TODO: change this
DATA_DIR = PROJECT_ROOT / "data"               # extracted dataset root
FORCE_EXTRACT = False

# Optional runtime copy is faster in Colab when images are on Drive.
USE_RUNTIME_COPY = True
RUNTIME_DATA_DIR = Path("/content/image_captioning_data")
FORCE_RUNTIME_RESYNC = False

# ----- File/path placeholders -----
# Leave as None to auto-search common names, then update after inspecting printed files.
TRAIN_ANNOTATION_PATH = None       # e.g. DATA_DIR / "train.json"
VAL_ANNOTATION_PATH = None         # e.g. DATA_DIR / "val.json"
TEST_ANNOTATION_PATH = None        # usually optional
SAMPLE_SUBMISSION_PATH = None      # e.g. DATA_DIR / "sample_submission.csv"

TRAIN_IMAGE_DIRS = []              # e.g. [DATA_DIR / "train" / "train"]
VAL_IMAGE_DIRS = []                # e.g. [DATA_DIR / "val" / "val"]
TEST_IMAGE_DIRS = []               # e.g. [DATA_DIR / "test" / "test"]

# ----- Annotation schema -----
# Supported annotation formats:
# - JSON dict: {"image.jpg": ["caption1", "caption2"]}
# - JSON list: [{"image": "...", "caption": "..."}, ...]
# - CSV with image and caption columns
IMAGE_COLUMN = None                # e.g. "image", "filename", "image_id"
CAPTION_COLUMN = None              # e.g. "caption", "labels", "text"
ID_COLUMN = None                   # from sample submission; inferred if None
SUBMISSION_CAPTION_COLUMN = None   # from sample submission; inferred if None

# ----- Language/model generation -----
CAPTION_LANGUAGE = "Thai"          # TODO: change language if needed
PROMPT = "<image> caption in Thai" # TODO: adapt to task/model
FALLBACK_CAPTION = "image caption"

# Strong but heavy default from the reference workflow. For faster baselines, try:
# "Salesforce/blip-image-captioning-base" or another captioning model supported by AutoProcessor.
MODEL_ID = "google/paligemma2-3b-pt-224"
MODEL_CLASS = "paligemma"          # "paligemma", "auto_vision2seq", or "auto_causallm"
LOCAL_FILES_ONLY = False
HF_CACHE_DIR = PROJECT_ROOT / "hf_cache"

# ----- Baseline inference -----
RUN_BASELINE_INFERENCE = True
BASELINE_MODEL_PATH = MODEL_ID
BASELINE_ADAPTER_PATH = None       # optional LoRA adapter for prediction
BATCH_SIZE = 2
MAX_NEW_TOKENS = 48
NUM_BEAMS = 4
LENGTH_PENALTY = 1.0
NO_REPEAT_NGRAM_SIZE = 3
EARLY_STOPPING = True

# ----- Training controls -----
RUN_TRAINING = False               # default off for fast preset runs
SMOKE_TEST = True
MAX_TRAIN_ROWS = 100 if SMOKE_TEST else None
MAX_VAL_IMAGES = 50 if SMOKE_TEST else None
TRAIN_ONE_CAPTION_PER_IMAGE = True
OUTPUT_DIR = PROJECT_ROOT / "outputs"
ADAPTER_OUTPUT_DIR = OUTPUT_DIR / "caption_lora_adapter"

# QLoRA/LoRA settings
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "auto"       # "auto" or list of module names
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1.0 if SMOKE_TEST else 3.0
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
FP16 = False
BF16 = True
GRADIENT_CHECKPOINTING = True

# ----- Metric / output -----
EVAL_METRIC = "sacrebleu"
OUTPUT_PATH = OUTPUT_DIR / "submission.csv"

In [ ]:
# ============================================================
# 4. Dataset extraction and file inspection
# ============================================================
if USE_GOOGLE_DRIVE and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped or unavailable:", type(e).__name__, e)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_zip_if_needed(zip_path: Path, out_dir: Path, force: bool = False):
    zip_path = Path(zip_path)
    out_dir = Path(out_dir)
    if not zip_path.exists():
        print("Zip not found. If data is already extracted, this is OK:", zip_path)
        return
    if force and out_dir.exists():
        shutil.rmtree(out_dir)
    if out_dir.exists() and any(out_dir.iterdir()):
        print("Using existing extracted dir:", out_dir)
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    print("Extracted:", zip_path, "->", out_dir)

extract_zip_if_needed(DATA_ZIP_PATH, DATA_DIR, FORCE_EXTRACT)

def sync_runtime_copy(src: Path, dst: Path, force: bool = False):
    if not USE_RUNTIME_COPY:
        return src
    src, dst = Path(src), Path(dst)
    if force and dst.exists():
        shutil.rmtree(dst)
    if dst.exists() and any(dst.iterdir()):
        print("Using existing runtime copy:", dst)
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        subprocess.run(["rsync", "-a", "--delete", f"{src}/", f"{dst}/"], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Runtime copy ready:", dst)
    return dst

ACTIVE_DATA_DIR = sync_runtime_copy(DATA_DIR, RUNTIME_DATA_DIR, FORCE_RUNTIME_RESYNC) if DATA_DIR.exists() else DATA_DIR

def inspect_files(root: Path, max_files: int = 120):
    root = Path(root)
    if not root.exists():
        print("Data dir does not exist yet:", root)
        return []
    files = sorted([p for p in root.rglob("*") if p.is_file()])
    print("file count:", len(files))
    for p in files[:max_files]:
        print(p.relative_to(root), p.stat().st_size)
    if len(files) > max_files:
        print("... truncated")
    return files

all_files = inspect_files(ACTIVE_DATA_DIR)

In [ ]:
# ============================================================
# 5. Auto-detect common files and load data
# ============================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def first_existing(candidates):
    for p in candidates:
        if p is not None and Path(p).exists():
            return Path(p)
    return None

def auto_find_annotation(root: Path, kind: str):
    patterns = {
        "train": ["*train*.json", "*train*.csv", "*annotations*.json", "*caption*.json"],
        "val": ["*val*.json", "*valid*.json", "*val*.csv", "*valid*.csv"],
        "sample": ["*sample_submission*.csv", "*submission*.csv"],
        "test": ["*test*.json", "*test*.csv"],
    }
    hits = []
    for pat in patterns[kind]:
        hits.extend(root.rglob(pat))
    return sorted(set(hits), key=lambda p: len(str(p)))[0] if hits else None

TRAIN_ANNOTATION_PATH = first_existing([TRAIN_ANNOTATION_PATH]) or auto_find_annotation(ACTIVE_DATA_DIR, "train")
VAL_ANNOTATION_PATH = first_existing([VAL_ANNOTATION_PATH]) or auto_find_annotation(ACTIVE_DATA_DIR, "val")
SAMPLE_SUBMISSION_PATH = first_existing([SAMPLE_SUBMISSION_PATH]) or auto_find_annotation(ACTIVE_DATA_DIR, "sample")
TEST_ANNOTATION_PATH = first_existing([TEST_ANNOTATION_PATH]) or auto_find_annotation(ACTIVE_DATA_DIR, "test")

print("TRAIN_ANNOTATION_PATH:", TRAIN_ANNOTATION_PATH)
print("VAL_ANNOTATION_PATH:", VAL_ANNOTATION_PATH)
print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("TEST_ANNOTATION_PATH:", TEST_ANNOTATION_PATH)

def find_image_dirs(root: Path):
    image_files = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    by_parent = Counter(p.parent for p in image_files)
    return [p for p, _ in by_parent.most_common()]

detected_image_dirs = find_image_dirs(ACTIVE_DATA_DIR)
print("Detected image dirs:")
for d in detected_image_dirs[:20]:
    print(" ", d, "images:", len([p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTS]))

if not TRAIN_IMAGE_DIRS:
    TRAIN_IMAGE_DIRS = [d for d in detected_image_dirs if "train" in str(d).lower()][:3]
if not VAL_IMAGE_DIRS:
    VAL_IMAGE_DIRS = [d for d in detected_image_dirs if any(x in str(d).lower() for x in ["val", "valid"])][:3]
if not TEST_IMAGE_DIRS:
    TEST_IMAGE_DIRS = [d for d in detected_image_dirs if "test" in str(d).lower()][:3]

print("TRAIN_IMAGE_DIRS:", TRAIN_IMAGE_DIRS)
print("VAL_IMAGE_DIRS:", VAL_IMAGE_DIRS)
print("TEST_IMAGE_DIRS:", TEST_IMAGE_DIRS)

In [ ]:
# ============================================================
# 6. Annotation loaders
# ============================================================
def read_json_or_csv(path: Path):
    path = Path(path)
    if path.suffix.lower() == ".json":
        return json.loads(path.read_text(encoding="utf-8"))
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported annotation file: {path}")

def infer_image_caption_columns(df: pd.DataFrame):
    image_candidates = [c for c in df.columns if any(k in c.lower() for k in ["image", "file", "filename", "path", "id"])]
    caption_candidates = [c for c in df.columns if any(k in c.lower() for k in ["caption", "text", "label", "description"])]
    image_col = IMAGE_COLUMN or (image_candidates[0] if image_candidates else df.columns[0])
    caption_col = CAPTION_COLUMN or (caption_candidates[0] if caption_candidates else (df.columns[1] if len(df.columns) > 1 else None))
    return image_col, caption_col

def annotation_to_records(obj, split: str):
    rows = []
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, dict):
        for image_rel, captions in obj.items():
            if isinstance(captions, str):
                captions = [captions]
            if not isinstance(captions, list):
                captions = [str(captions)]
            rows.append({"split": split, "image_rel": str(image_rel), "captions": [str(c) for c in captions if str(c).strip()]})
        return pd.DataFrame(rows)

    if isinstance(obj, list):
        df = pd.DataFrame(obj)
    elif isinstance(obj, pd.DataFrame):
        df = obj.copy()
    else:
        raise ValueError(f"Unsupported annotation object: {type(obj)}")

    image_col, caption_col = infer_image_caption_columns(df)
    if caption_col is None:
        df["captions"] = [[] for _ in range(len(df))]
    else:
        df["captions"] = df[caption_col].fillna("").astype(str).map(lambda x: [x] if x.strip() else [])
    df["image_rel"] = df[image_col].astype(str)
    df["split"] = split
    return df[["split", "image_rel", "captions"]].copy()

def load_annotation(path, split):
    if path is None:
        return pd.DataFrame(columns=["split", "image_rel", "captions"])
    return annotation_to_records(read_json_or_csv(Path(path)), split)

train_images_df = load_annotation(TRAIN_ANNOTATION_PATH, "train")
val_images_df = load_annotation(VAL_ANNOTATION_PATH, "val")
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH else pd.DataFrame()

if not sample_sub.empty:
    ID_COLUMN = ID_COLUMN or sample_sub.columns[0]
    SUBMISSION_CAPTION_COLUMN = SUBMISSION_CAPTION_COLUMN or sample_sub.columns[-1]

print("train_images_df:", train_images_df.shape)
print("val_images_df:", val_images_df.shape)
print("sample_sub:", sample_sub.shape)
display(train_images_df.head())
display(val_images_df.head())
display(sample_sub.head())

In [ ]:
# ============================================================
# 7. Image path resolution and basic overview
# ============================================================
def resolve_image_path(image_rel: str, image_dirs: list[Path]):
    p = Path(image_rel)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    candidates.append(ACTIVE_DATA_DIR / image_rel)
    for d in image_dirs:
        candidates.append(Path(d) / image_rel)
        candidates.append(Path(d) / Path(image_rel).name)
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]

def attach_image_paths(df: pd.DataFrame, dirs: list[Path]):
    if df.empty:
        return df.copy()
    out = df.copy()
    out["image_path"] = out["image_rel"].map(lambda x: resolve_image_path(x, dirs))
    out["image_exists"] = out["image_path"].map(lambda p: Path(p).exists())
    return out

train_images_df = attach_image_paths(train_images_df, TRAIN_IMAGE_DIRS)
val_images_df = attach_image_paths(val_images_df, VAL_IMAGE_DIRS or TRAIN_IMAGE_DIRS)

print("train images exist:", train_images_df["image_exists"].mean() if not train_images_df.empty else None)
print("val images exist:", val_images_df["image_exists"].mean() if not val_images_df.empty else None)

def flatten_caption_rows(image_df: pd.DataFrame):
    rows = []
    for row in image_df.itertuples(index=False):
        for i, caption in enumerate(row.captions):
            rows.append({
                "row_id": f"{row.image_rel}__ref{i}",
                "split": row.split,
                "image_rel": row.image_rel,
                "image_path": row.image_path,
                "caption": caption,
                "caption_idx": i,
                "image_exists": row.image_exists,
            })
    return pd.DataFrame(rows)

train_rows_df = flatten_caption_rows(train_images_df)
val_rows_df = flatten_caption_rows(val_images_df)

print("train caption rows:", train_rows_df.shape)
print("val caption rows:", val_rows_df.shape)
display(train_rows_df.head())

In [ ]:
# ============================================================
# 8. Basic EDA
# ============================================================
if not train_rows_df.empty:
    train_rows_df["char_len"] = train_rows_df["caption"].astype(str).str.len()
    train_rows_df["word_like_len"] = train_rows_df["caption"].astype(str).str.split().map(len)
    display(train_rows_df[["char_len", "word_like_len"]].describe())

    plt.figure(figsize=(10, 4))
    sns.histplot(train_rows_df["char_len"], bins=40)
    plt.title("Caption character length distribution")
    plt.show()

    print("Empty captions:", int((train_rows_df["caption"].str.strip() == "").sum()))
    print("Duplicate image-caption rows:", int(train_rows_df.duplicated(["image_rel", "caption"]).sum()))

def show_image_samples(df: pd.DataFrame, n: int = 6):
    if df.empty:
        print("No images to show.")
        return
    sample = df[df["image_exists"]].sample(min(n, df["image_exists"].sum()), random_state=RANDOM_STATE)
    cols = 3
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, row in zip(axes, sample.itertuples(index=False)):
        img = Image.open(row.image_path).convert("RGB")
        ax.imshow(img)
        cap = row.captions[0] if getattr(row, "captions", []) else ""
        ax.set_title(str(cap)[:80])
        ax.axis("off")
    for ax in axes[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_image_samples(train_images_df, n=6)

In [ ]:
# ============================================================
# 9. Preprocessing and feature engineering
# ============================================================
def clean_caption(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

if not train_rows_df.empty:
    train_rows_df["caption"] = train_rows_df["caption"].map(clean_caption)
if not val_rows_df.empty:
    val_rows_df["caption"] = val_rows_df["caption"].map(clean_caption)

if TRAIN_ONE_CAPTION_PER_IMAGE and not train_rows_df.empty:
    train_fit_df = (
        train_rows_df[train_rows_df["image_exists"]]
        .groupby("image_rel", group_keys=False)
        .sample(n=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
else:
    train_fit_df = train_rows_df[train_rows_df["image_exists"]].reset_index(drop=True)

val_eval_images_df = val_images_df[val_images_df["image_exists"]].reset_index(drop=True)

if MAX_TRAIN_ROWS is not None and len(train_fit_df) > MAX_TRAIN_ROWS:
    train_fit_df = train_fit_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
if MAX_VAL_IMAGES is not None and len(val_eval_images_df) > MAX_VAL_IMAGES:
    val_eval_images_df = val_eval_images_df.sample(MAX_VAL_IMAGES, random_state=RANDOM_STATE).reset_index(drop=True)

print("train_fit_df:", train_fit_df.shape)
print("val_eval_images_df:", val_eval_images_df.shape)

In [ ]:
# ============================================================
# 10. Train/validation split fallback
# ============================================================
# If no validation split exists, create an image-level split from training images.
if val_eval_images_df.empty and not train_images_df.empty:
    from sklearn.model_selection import train_test_split
    unique_images = train_images_df[train_images_df["image_exists"]].reset_index(drop=True)
    tr_img, va_img = train_test_split(unique_images, test_size=0.15, random_state=RANDOM_STATE)
    train_images_df = tr_img.reset_index(drop=True)
    val_eval_images_df = va_img.reset_index(drop=True)
    train_rows_df = flatten_caption_rows(train_images_df)
    val_rows_df = flatten_caption_rows(val_eval_images_df)
    train_fit_df = train_rows_df.groupby("image_rel", group_keys=False).sample(n=1, random_state=RANDOM_STATE).reset_index(drop=True)
    print("Created validation split:", train_fit_df.shape, val_eval_images_df.shape)

In [ ]:
# ============================================================
# 11. Model loading helpers
# ============================================================
def get_quant_config(load_in_4bit: bool = LOAD_IN_4BIT):
    if not load_in_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )

def load_processor(model_id: str = MODEL_ID):
    return AutoProcessor.from_pretrained(
        model_id,
        cache_dir=str(HF_CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
    )

def load_caption_model(model_id: str = MODEL_ID, adapter_path=None, for_training: bool = False):
    quant_config = get_quant_config(LOAD_IN_4BIT)
    common_kwargs = dict(
        cache_dir=str(HF_CACHE_DIR),
        local_files_only=LOCAL_FILES_ONLY,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=torch.bfloat16 if BF16 else torch.float16,
        quantization_config=quant_config,
    )
    common_kwargs = {k: v for k, v in common_kwargs.items() if v is not None}

    if MODEL_CLASS == "paligemma" and PaliGemmaForConditionalGeneration is not None:
        model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, **common_kwargs)
    elif MODEL_CLASS == "auto_causallm":
        model = AutoModelForCausalLM.from_pretrained(model_id, **common_kwargs)
    else:
        if AutoModelForVision2Seq is None:
            raise ImportError("AutoModelForVision2Seq is unavailable in this transformers version. Use MODEL_CLASS='paligemma' or 'auto_causallm'.")
        model = AutoModelForVision2Seq.from_pretrained(model_id, **common_kwargs)

    if adapter_path is not None:
        if PeftModel is None:
            raise ImportError("peft is required to load adapters.")
        model = PeftModel.from_pretrained(model, adapter_path)

    if for_training and GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    return model

def clean_generated_text(text: str, prompt: str = PROMPT) -> str:
    text = str(text).strip()
    if prompt and text.startswith(prompt):
        text = text[len(prompt):].strip()
    for prefix in [PROMPT, "caption in Thai", "caption:", "Caption:"]:
        if prefix and text.startswith(prefix):
            text = text[len(prefix):].strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"\s+", " ", text)
    return text or FALLBACK_CAPTION

In [ ]:
# ============================================================
# 12. Baseline model inference
# ============================================================
def generate_captions(image_paths, processor, model, prompt=PROMPT, batch_size=BATCH_SIZE, **gen_kwargs):
    model.eval()
    outputs = []
    device = next(model.parameters()).device
    for start in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[start:start + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]
        prompts = [prompt] * len(images)
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=gen_kwargs.get("max_new_tokens", MAX_NEW_TOKENS),
                num_beams=gen_kwargs.get("num_beams", NUM_BEAMS),
                length_penalty=gen_kwargs.get("length_penalty", LENGTH_PENALTY),
                no_repeat_ngram_size=gen_kwargs.get("no_repeat_ngram_size", NO_REPEAT_NGRAM_SIZE),
                early_stopping=gen_kwargs.get("early_stopping", EARLY_STOPPING),
            )
        texts = processor.batch_decode(generated, skip_special_tokens=True)
        outputs.extend([clean_generated_text(t, prompt) for t in texts])
    return outputs

baseline_processor = None
baseline_model = None

if RUN_BASELINE_INFERENCE:
    baseline_processor = load_processor(BASELINE_MODEL_PATH)
    baseline_model = load_caption_model(BASELINE_MODEL_PATH, adapter_path=BASELINE_ADAPTER_PATH, for_training=False)
    print("Baseline model loaded.")
else:
    print("Baseline inference disabled.")

In [ ]:
# ============================================================
# 13. Validation and metric calculation
# ============================================================
def score_bleu(preds: list[str], refs_map: dict[str, list[str]], image_rels: list[str]) -> dict:
    refs = [refs_map.get(image_rel, [FALLBACK_CAPTION]) for image_rel in image_rels]
    max_refs = max(len(r) for r in refs) if refs else 0
    refs_by_index = []
    for i in range(max_refs):
        refs_by_index.append([r[i] if i < len(r) else r[0] for r in refs])
    bleu = sacrebleu.corpus_bleu(preds, refs_by_index) if preds else None
    return {"sacrebleu": bleu.score if bleu else None}

val_preds = []
val_scores = {}

if RUN_BASELINE_INFERENCE and baseline_model is not None and not val_eval_images_df.empty:
    val_paths = val_eval_images_df["image_path"].tolist()
    val_rels = val_eval_images_df["image_rel"].tolist()
    val_refs = dict(zip(val_eval_images_df["image_rel"], val_eval_images_df["captions"]))
    val_preds = generate_captions(val_paths, baseline_processor, baseline_model, batch_size=BATCH_SIZE)
    val_scores = score_bleu(val_preds, val_refs, val_rels)
    print("Validation:", val_scores)
    display(pd.DataFrame({"image_rel": val_rels, "prediction": val_preds, "refs": [val_refs[x] for x in val_rels]}).head(20))
else:
    print("Skipping validation generation.")

In [ ]:
# ============================================================
# 14. Test prediction
# ============================================================
def build_test_df():
    if sample_sub.empty:
        # Fallback: use detected test image files.
        test_files = []
        for d in TEST_IMAGE_DIRS:
            test_files.extend(sorted([p for p in Path(d).glob("*") if p.suffix.lower() in IMAGE_EXTS]))
        return pd.DataFrame({
            ID_COLUMN or "id": [p.stem for p in test_files],
            "image_path": test_files,
            "image_rel": [p.name for p in test_files],
        })

    df = sample_sub.copy()
    id_col = ID_COLUMN or df.columns[0]
    # Try to resolve sample IDs as file names or stems.
    test_files = []
    for d in TEST_IMAGE_DIRS:
        test_files.extend(sorted([p for p in Path(d).glob("*") if p.suffix.lower() in IMAGE_EXTS]))
    by_name = {p.name: p for p in test_files}
    by_stem = {p.stem: p for p in test_files}

    def resolve_from_id(x):
        x = str(x)
        if x in by_name:
            return by_name[x]
        if x in by_stem:
            return by_stem[x]
        for ext in IMAGE_EXTS:
            if x + ext in by_name:
                return by_name[x + ext]
        return resolve_image_path(x, TEST_IMAGE_DIRS)

    df["image_path"] = df[id_col].map(resolve_from_id)
    df["image_rel"] = df[id_col].astype(str)
    df["image_exists"] = df["image_path"].map(lambda p: Path(p).exists())
    return df

test_df = build_test_df()
print("test_df:", test_df.shape)
print("test image exists:", test_df["image_exists"].mean() if "image_exists" in test_df else None)
display(test_df.head())

In [ ]:
# ============================================================
# 15. Submission file generation
# ============================================================
submission = sample_sub.copy() if not sample_sub.empty else pd.DataFrame()

if RUN_BASELINE_INFERENCE and baseline_model is not None and not test_df.empty:
    test_paths = test_df["image_path"].tolist()
    test_preds = generate_captions(test_paths, baseline_processor, baseline_model, batch_size=BATCH_SIZE)

    if submission.empty:
        submission = pd.DataFrame({
            ID_COLUMN or "id": test_df[ID_COLUMN or "id"],
            SUBMISSION_CAPTION_COLUMN or "caption": test_preds,
        })
    else:
        submission[SUBMISSION_CAPTION_COLUMN] = test_preds

    submission[SUBMISSION_CAPTION_COLUMN] = submission[SUBMISSION_CAPTION_COLUMN].map(clean_caption).replace("", FALLBACK_CAPTION)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    print("Saved:", OUTPUT_PATH)
    display(submission.head())
    print("rows:", len(submission), "blank captions:", int((submission[SUBMISSION_CAPTION_COLUMN].astype(str).str.strip() == "").sum()))
else:
    print("Skipping test prediction. Set RUN_BASELINE_INFERENCE=True after config is ready.")

## Optional Improvement Ideas

Each section below is guarded by a flag so the baseline stays fast by default.

In [ ]:
# ============================================================
# Optional A. LoRA / QLoRA fine-tuning
# Use when you have image-caption training pairs and a GPU runtime.
# Tradeoff: stronger leaderboard potential, but slow and memory-sensitive.
# ============================================================
RUN_LORA_FINETUNE = RUN_TRAINING

class CaptionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, prompt: str = PROMPT):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.prompt = prompt

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        text = f"{self.prompt} {row['caption']}"
        enc = self.processor(text=text, images=image, return_tensors="pt", padding="max_length", truncation=True)
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = item["input_ids"].clone()
        return item

def infer_lora_targets(model):
    if LORA_TARGET_MODULES != "auto":
        return LORA_TARGET_MODULES
    names = [name for name, _ in model.named_modules()]
    common = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    found = sorted({c for c in common if any(name.endswith(c) for name in names)})
    return found or ["q_proj", "v_proj"]

if RUN_LORA_FINETUNE:
    assert len(train_fit_df) > 0, "No training rows found."
    assert LoraConfig is not None, "peft is required."
    processor = load_processor(MODEL_ID)
    model = load_caption_model(MODEL_ID, for_training=True)
    if LOAD_IN_4BIT and prepare_model_for_kbit_training is not None:
        model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=infer_lora_targets(model),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    train_ds = CaptionDataset(train_fit_df, processor)
    args = TrainingArguments(
        output_dir=str(ADAPTER_OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        logging_steps=20,
        save_strategy="epoch",
        report_to="none",
        fp16=FP16,
        bf16=BF16 and torch.cuda.is_available(),
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds)
    trainer.train()
    model.save_pretrained(ADAPTER_OUTPUT_DIR)
    processor.save_pretrained(ADAPTER_OUTPUT_DIR)
    print("Saved adapter:", ADAPTER_OUTPUT_DIR)
else:
    print("LoRA fine-tuning disabled.")

In [ ]:
# ============================================================
# Optional B. Generation hyperparameter tuning
# Use when validation references exist.
# Tradeoff: cheap and often improves BLEU, but may overfit public validation style.
# ============================================================
RUN_GENERATION_TUNING = False

if RUN_GENERATION_TUNING:
    assert baseline_model is not None and not val_eval_images_df.empty
    tune_rows = val_eval_images_df.sample(min(200, len(val_eval_images_df)), random_state=RANDOM_STATE)
    refs_map = dict(zip(val_eval_images_df["image_rel"], val_eval_images_df["captions"]))
    grid = [
        {"name": "beam4_len48_lp1.0", "max_new_tokens": 48, "num_beams": 4, "length_penalty": 1.0, "no_repeat_ngram_size": 3},
        {"name": "beam5_len56_lp0.9", "max_new_tokens": 56, "num_beams": 5, "length_penalty": 0.9, "no_repeat_ngram_size": 3},
        {"name": "beam3_len40_lp1.1", "max_new_tokens": 40, "num_beams": 3, "length_penalty": 1.1, "no_repeat_ngram_size": 2},
    ]
    results = []
    for cfg in grid:
        preds = generate_captions(tune_rows["image_path"].tolist(), baseline_processor, baseline_model, batch_size=BATCH_SIZE, **cfg)
        score = score_bleu(preds, refs_map, tune_rows["image_rel"].tolist())
        results.append({**cfg, **score})
    tune_results_df = pd.DataFrame(results).sort_values("sacrebleu", ascending=False)
    display(tune_results_df)
else:
    print("Generation tuning disabled.")

In [ ]:
# ============================================================
# Optional C. K-fold validation skeleton
# Use when the dataset is small and validation scores are noisy.
# Tradeoff: expensive for model fine-tuning; best used for evaluation or adapters.
# ============================================================
RUN_KFOLD = False

if RUN_KFOLD:
    from sklearn.model_selection import KFold
    unique = train_images_df[train_images_df["image_exists"]].reset_index(drop=True)
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(unique), 1):
        tr_images = unique.iloc[tr_idx].reset_index(drop=True)
        va_images = unique.iloc[va_idx].reset_index(drop=True)
        print("fold", fold, "train images", len(tr_images), "val images", len(va_images))
        # TODO: train or load one adapter per fold, then generate va predictions.
        fold_rows.append({"fold": fold, "train_images": len(tr_images), "val_images": len(va_images)})
    display(pd.DataFrame(fold_rows))
else:
    print("K-fold disabled.")

In [ ]:
# ============================================================
# Optional D. Test-time augmentation / prompt ensembling
# Use when generation is unstable.
# Tradeoff: slower inference, sometimes better style robustness.
# ============================================================
RUN_TTA = False

if RUN_TTA:
    assert baseline_model is not None
    prompts = [
        PROMPT,
        "<image> describe this image in Thai",
        "<image> write a concise Thai caption",
    ]
    sample_paths = test_df["image_path"].tolist()
    all_prompt_preds = []
    for prompt in prompts:
        preds = generate_captions(sample_paths, baseline_processor, baseline_model, prompt=prompt, batch_size=BATCH_SIZE)
        all_prompt_preds.append(preds)
    # Simple choice: use first prompt. TODO: replace with reranking by validation-tuned heuristic.
    tta_preds = all_prompt_preds[0]
    tta_submission = sample_sub.copy()
    tta_submission[SUBMISSION_CAPTION_COLUMN] = tta_preds
    tta_path = OUTPUT_PATH.with_name("submission_tta.csv")
    tta_submission.to_csv(tta_path, index=False)
    print("Saved:", tta_path)
else:
    print("TTA disabled.")

In [ ]:
# ============================================================
# Optional E. Adapter/model ensembling
# Use after training multiple checkpoints.
# Tradeoff: slower and needs more storage; usually stable if models are diverse.
# ============================================================
RUN_ENSEMBLE = False
ENSEMBLE_ADAPTERS = [
    # TODO: add adapter paths, e.g. PROJECT_ROOT / "outputs" / "adapter_fold1"
]

if RUN_ENSEMBLE:
    ensemble_preds = []
    for adapter_path in ENSEMBLE_ADAPTERS:
        processor = load_processor(MODEL_ID)
        model = load_caption_model(MODEL_ID, adapter_path=adapter_path)
        preds = generate_captions(test_df["image_path"].tolist(), processor, model, batch_size=BATCH_SIZE)
        ensemble_preds.append(preds)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    # TODO: implement caption reranking. Default: first model.
    final_preds = ensemble_preds[0]
    ens_submission = sample_sub.copy()
    ens_submission[SUBMISSION_CAPTION_COLUMN] = final_preds
    ens_path = OUTPUT_PATH.with_name("submission_ensemble.csv")
    ens_submission.to_csv(ens_path, index=False)
    print("Saved:", ens_path)
else:
    print("Ensemble disabled.")

In [ ]:
# ============================================================
# Optional F. Error analysis
# Use after validation predictions exist.
# Tradeoff: manual inspection time, but often reveals prompt/length/data issues.
# ============================================================
RUN_ERROR_ANALYSIS = False

if RUN_ERROR_ANALYSIS and val_preds:
    analysis_df = pd.DataFrame({
        "image_rel": val_eval_images_df["image_rel"].tolist(),
        "image_path": val_eval_images_df["image_path"].tolist(),
        "prediction": val_preds,
        "references": val_eval_images_df["captions"].tolist(),
    })
    analysis_df["pred_len"] = analysis_df["prediction"].str.len()
    analysis_df["ref_len_mean"] = analysis_df["references"].map(lambda xs: np.mean([len(x) for x in xs]) if xs else 0)
    analysis_df["len_gap"] = (analysis_df["pred_len"] - analysis_df["ref_len_mean"]).abs()
    display(analysis_df.sort_values("len_gap", ascending=False).head(30))
else:
    print("Error analysis disabled or no validation predictions.")

In [ ]:
# ============================================================
# Optional G. Pseudo-labeling skeleton
# Use when test distribution differs and validation is stable.
# Tradeoff: can improve domain fit, but bad pseudo-labels reinforce errors.
# ============================================================
RUN_PSEUDO_LABELING = False

if RUN_PSEUDO_LABELING:
    assert OUTPUT_PATH.exists(), "Generate baseline submission first."
    pseudo = pd.read_csv(OUTPUT_PATH)
    pseudo_train = test_df.copy()
    pseudo_train["caption"] = pseudo[SUBMISSION_CAPTION_COLUMN].values
    pseudo_train = pseudo_train.rename(columns={"image_rel": "image_rel"})
    pseudo_train["row_id"] = pseudo_train[ID_COLUMN or sample_sub.columns[0]].astype(str)
    pseudo_train["image_exists"] = pseudo_train["image_path"].map(lambda p: Path(p).exists())
    display(pseudo_train.head())
    # TODO: concatenate a filtered subset into train_fit_df and run LoRA fine-tuning again.
else:
    print("Pseudo-labeling disabled.")

## Debug Checklist

Before submitting:

1. Confirm `DATA_DIR`, annotation paths, and image directories are correct.
2. Confirm train/validation/test image existence rates are near `1.0`.
3. Run with `SMOKE_TEST = True` first.
4. Inspect sample generated captions manually.
5. Confirm submission row count equals sample submission row count.
6. Confirm no blank captions.
7. Save the exact config/model/checkpoint used for each submission.

If the notebook fails:

- If model download fails, set `LOCAL_FILES_ONLY = False` or authenticate with Hugging Face.
- If GPU memory fails, reduce batch size, enable 4-bit, reduce image size/model size, or disable beams.
- If generated captions echo the prompt, improve `clean_generated_text`.
- If validation BLEU is unstable, tune generation on a held-out subset and inspect examples.